In [ ]:
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png r::r-raster conda-forge::r-fnn")
system("conda install -y conda-forge::r-cluster conda-forge::r-remotes conda-forge::r-devtools")
system("conda install -y conda-forge::r-factominer conda-forge::r-caret conda-forge::r-factoextra conda-forge::r-rlang")
system("conda install -y conda-forge::r-geojsonio")


In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(terra)     
library(dplyr)
        
library(jsonlite) 
library(utils)
library(ggplot2)

library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

library(data.table)
# library(FNN)

library(cluster)
# library(bioregion)

# library(rlang)
# library(factoextra)

library(sf)
library(geojsonio)

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"Data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.filename <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
    } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename, type = "gz"){
    filename.length <- nchar(filename)
    print(filename, filename.length)

    start <- 1

    if(type == "gz"){
        # Case Gunzip
        end <- filename.length-3
    }
    else{
        # Case Unzip
        end <- filename.length-4
    }

    unzip.filename <- substr(filename,start, end)
    print(unzip.filename)

    
    # Case gunzip
    if(!file.exists(unzip.filename)){
        if(type == "gz"){
            R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
            }
        else{
            utils::unzip(filename, overwrite=FALSE )
            }
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}


In [ ]:
gdb_path_zip <- "Geomorphology.gdb.zip"
gdb_path <- unzip.file(gdb_path_zip, type = "zip")

# gdb_path <- move.file(gdb_path, paste(dataDir,"Geomorphology.gdb",sep = "/"))


In [ ]:
# https://gis.stackexchange.com/questions/426282/load-gdb-directory-into-r-using-simple-features-package
model <- sf::st_read(dsn = gdb_path)

# Select Seamount, Seamount Ridge and Canyon
seamounts <- model$Feature[model$Feature %in% c("Seamount", "Seamount Ridge")]
canyons <- model$Feature[model$Feature %in% c("Canyon")]
others <- model$Feature[model$Feature %in% c("Coastal/Shelf Terrane", "Cross Shelf Valley",
                                            "Ridge", "Shelf Deep", "Trough Mouth Fan", 
                                            "Contourite Feature", "Plateau", "")]


In [ ]:
model2 <- model %>%
  mutate(coastal.terrane = ifelse(Feature == "Coastal/Shelf Terrane", yes = 1, no = 0),
         cross.shelf.valley = ifelse(Feature == "Cross Shelf Valley", yes = 1, no = 0),
         margin.ridges = ifelse(Feature == "Margin Ridges", yes = 1, no = 0),
         shelf.bank = ifelse(Feature == "Bank", yes = 1, no = 0),
         shelf.deep = ifelse(Feature == "Shelf Deep", yes = 1, no = 0), 
         trough.mouth.fan = ifelse(Feature == "Trough Mouth Fan", yes = 1, no = 0),
         contourite.ft = ifelse(Feature == "Contourite Feature", yes = 1, no = 0),
         plateau = ifelse(Feature == "Plateau", yes = 1, no = 0), 
         ridge.ft = ifelse(Feature == "Ridge", yes = 1, no = 0)
         )
         

In [ ]:
head(model2$Feature)

In [ ]:
print(model)


In [ ]:
unique(model$Feature)
table(model$Feature)
head(model$Shape_Area,5)
head(model$Shape_Length,5)


In [ ]:
model$Shape

In [ ]:
## HELP
# https://r.geocompx.org/solutions/read-write.html
# https://github.com/highered-esricanada/r-arcgis-tutorials/blob/master/3-R-ArcGIS-Scripting.pdf
# = https://esricanada-ce.github.io/r-arcgis-tutorials/3-R-ArcGIS-Scripting.pdf


In [ ]:
# View
plot(model)


In [ ]:
# List layers inside the geodatabase
gbd.layers <- st_layers(gdb_path)
print(gbd.layers)

In [ ]:
# Analysis
summary(model)

In [ ]:
# Check for the projection
st_crs(model)

In [ ]:
# If it's not WGS84 (EPSG:4326)
data <- st_transform(model, 4326)
data <- st_make_valid(data)

# ⚠️ If geometry is NOT points
#🔸 For polygons or lines
centroids <- st_centroid(data)
coords <- st_coordinates(centroids)

# Export to 
write.csv(st_drop_geometry(data), paste("outputs/collection/st_drop_geometry.csv"), row.names = FALSE)

In [ ]:
typeof(model)
typeof(model$Feature)
typeof(model$Shape)
typeof(model$Shape_Area)
typeof(model$Shape_Length)

In [ ]:
# ==============================================================================
# 1. COMPUTE DISTANCE TO CANYON
# ==============================================================================

# Split canyon vs other features
# Canyon = Cross Shelf Valley
canyons <- model[model$Feature == "Canyon", ]

# Compute distances
# Other features (or all, depending on your goal)
other <- model
dist_matrix <- st_distance(other, canyons)
min_dist <- apply(dist_matrix, 1, min)

# Extract minimum distance
other$dist_to_canyon <- as.numeric(min_dist)